In [1]:
"""B-4: Hopfield Network to store and recall 4 bipolar vectors."""
import numpy as np


In [2]:
def sign_bipolar(x: np.ndarray) -> np.ndarray:
    return np.where(x >= 0, 1, -1)

In [3]:
def pretty(pattern: np.ndarray, side: int = 3) -> str:
    chars = np.where(pattern.reshape(side, side) > 0, "1", "0")
    return "\n".join(" ".join(row) for row in chars)

In [4]:
class Hopfield:
    def __init__(self, patterns: np.ndarray):
        n = patterns.shape[1]
        self.W = np.zeros((n, n))
        for p in patterns:
            self.W += np.outer(p, p)
        np.fill_diagonal(self.W, 0)
        self.W /= n

    def recall(self, x: np.ndarray, steps: int = 10, seed: int = 1) -> np.ndarray:
        state = x.copy()
        rng = np.random.default_rng(seed)

        for _ in range(steps):
            for idx in rng.permutation(len(state)):
                net = np.dot(self.W[idx], state)
                state[idx] = 1 if net >= 0 else -1
        return state


In [5]:
def main() -> None:
    # Four 3x3 patterns (bipolar: -1/+1)
    patterns = np.array(
        [
            [1, 1, 1, 1, -1, 1, 1, 1, 1],   # Ring-like
            [1, -1, 1, -1, 1, -1, 1, -1, 1],
            [1, 1, -1, 1, -1, -1, 1, 1, -1],
            [1, -1, -1, 1, 1, -1, 1, -1, -1],
        ],
        dtype=int,
    )

    net = Hopfield(patterns)
    print("Stored 4 patterns in Hopfield network.\n")

    # Create noisy versions by flipping one bit from each pattern.
    noisy = patterns.copy()
    noisy[:, 4] *= -1

    correct = 0
    for i in range(4):
        recalled = net.recall(noisy[i], steps=15, seed=i + 10)
        match = np.array_equal(recalled, patterns[i])
        correct += int(match)

        print(f"Pattern {i + 1}")
        print("Noisy input:")
        print(pretty(noisy[i]))
        print("Recalled:")
        print(pretty(recalled))
        print("Expected:")
        print(pretty(patterns[i]))
        print(f"Match: {match}\n")

    acc = correct / 4 * 100
    print(f"Recall accuracy for 4 stored vectors: {acc:.2f}%")


In [6]:
if __name__ == "__main__":
    main()

Stored 4 patterns in Hopfield network.

Pattern 1
Noisy input:
1 1 1
1 1 1
1 1 1
Recalled:
1 1 1
1 0 1
1 1 1
Expected:
1 1 1
1 0 1
1 1 1
Match: True

Pattern 2
Noisy input:
1 0 1
0 0 0
1 0 1
Recalled:
1 0 1
0 1 0
1 0 1
Expected:
1 0 1
0 1 0
1 0 1
Match: True

Pattern 3
Noisy input:
1 1 0
1 1 0
1 1 0
Recalled:
1 1 0
1 0 0
1 1 0
Expected:
1 1 0
1 0 0
1 1 0
Match: True

Pattern 4
Noisy input:
1 0 0
1 0 0
1 0 0
Recalled:
1 1 0
1 0 0
1 1 0
Expected:
1 0 0
1 1 0
1 0 0
Match: False

Recall accuracy for 4 stored vectors: 75.00%
